# **SprintLab**


---


## Athlete Performance Analytics & Development Intelligence Platform

A comprehensive data analytics project focused on tracking, analyzing, and visualizing the long-term development of athletes trained under my coaching program. This project combines performance data, event specialization, and progression trends to evaluate athlete improvement over time through statistical analysis, interactive visualizations, and data-driven insights.


# **Core Objectives**

* Analyze multi-year athlete performance progression
* Identify trends across sprint, jump, and multi-event athletes
* Measure year-over-year development and performance growth
* Visualize athlete trajectories using interactive dashboards
* Explore relationships between training focus and competitive outcomes
* Build a scalable analytics pipeline for future predictive modeling and coaching insights



## Future Development

Planned future features include:
- Flask web dashboard
- SQL database integration
- Athlete profile pages
- Predictive performance modeling
- Automated data entry forms
- Training load analysis

## Technology Stack

- Python
- Pandas
- NumPy
- Plotly Express
- Matplotlib
- Seaborn
- Jupyter Notebook

In [74]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

In [97]:
df = pd.read_csv("/content/drive/MyDrive/SprintLab Analytics/data/SprintLab_Data_Sheet.csv")

## Dataset Overview

The dataset contains:
- athlete performance results
- sprint and field event marks
- meet information
- placement data
- seasonal performance history
- progression metrics

Each row represents a single athletic performance.

# Phase 1 - Data Cleaning & Feature Engineering

## Step 1 - Clean Column Names

In [98]:
def clean_column_names(df):
    """
    Clean column names
    """
    df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

    for name in df.columns:
        print(name)

    return df



athlete_id
name
gender
grade
athlete_class
event
raw_result
result_in_seconds
result_in_inches
result_in_meters
wind
placement
date
meet
atmosphere
race_type
season
performance_type


# Step 2 - Standardize All Categorical Data / Correct Datatypes


In [77]:
# Standardizing Categorical Data / Correct Datatypes
def clean_col_names(df):
    """
    remove empty space and lowercase column  names
    """

    text_cols = ["name", "gender", "event", "meet", "atmosphere", "race_type", "season", "performance_type"]

    for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

    df["event"] = df["event"].str.lower()

    return df



In [78]:
def convert_datatypes(df):
    """
    Convert datatypes of columns properly
    """

    category_cols = ["gender", "grade", "athlete_class", "event", "meet", "atmosphere", "race_type", "season", "performance_type"]

    for col in category_cols:
        df[col] = df[col].astype("category")

    return df





In [79]:
df["date"] = pd.to_datetime(df["date"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   athlete_id         101 non-null    int64         
 1   name               101 non-null    object        
 2   gender             101 non-null    category      
 3   grade              101 non-null    category      
 4   athlete_class      101 non-null    category      
 5   event              101 non-null    category      
 6   raw_result         101 non-null    object        
 7   result_in_seconds  49 non-null     float64       
 8   result_in_inches   52 non-null     float64       
 9   result_in_meters   52 non-null     float64       
 10  wind               17 non-null     float64       
 11  placement          101 non-null    int64         
 12  date               101 non-null    datetime64[ns]
 13  meet               101 non-null    category      
 14  atmosphere

In [80]:
# Check for duplicates

print(f"Duplicate results: {df.duplicated().sum()}")

Duplicate results: 0


# Step 3 - Event Grouping


In [81]:
# Event Grouping

track_events = ["60m", "60mh", "100m", "100mh", "110mh", "200m", "300mh", "400m", "800m", "1600m", "3200m"]
field_events = ["long jump", "triple jump", "high jump", "javelin", "pole vault", "shot put", "discus"]

def classify_event(event):

  if event in track_events:
    return "track"
  elif event in field_events:
    return "field"
  else:
    return "other"

df["event_type"] = df["event"].apply(classify_event).astype("category")

df.event_type.value_counts()

,count
event_type,
field,52
track,49


# Step 4 - Establish Improvement Direction

### Why Event Classification Matters

Track and field events require different analytical logic:
- lower sprint times represent improvement
- higher field marks represent improvement

This classification system allows progression algorithms to adapt dynamically based on event type.

In [82]:
# Establishing the improvement direction

def better_direction(event_type):
  if event_type == "track":
    return "lower"
  elif event_type == "field":
    return "higher"
  else:
    return "unknown"

df["better_direction"] = df["event_type"].apply(better_direction).astype("category")

df.better_direction.value_counts()

,count
better_direction,
higher,52
lower,49


# Step 5 - Universal Results Column

In [83]:
# Universal results column

df["result_value"] = df["result_in_seconds"].fillna(df["result_in_meters"])

df.sample(5)

,athlete_id,name,gender,grade,athlete_class,event,raw_result,result_in_seconds,result_in_inches,result_in_meters,...,placement,date,meet,atmosphere,race_type,season,performance_type,event_type,better_direction,result_value
76,2,Khiyon Lewis,Male,12,2024,triple jump,45-6.25,NaN,546.25,13.87,...,4,2024-04-13,Mountain Brook Invitational,Outdoor,Finals,2024 Outdoor,Official,field,higher,13.87
33,1,Harley McNeal,Female,11,2024,60m,7.94,7.94,NaN,NaN,...,5,2022-12-03,Magic City Invitational #2,Indoor,Finals,2023 Indoor,Official,track,lower,7.94
70,2,Khiyon Lewis,Male,12,2024,triple jump,44-0,NaN,528.00,13.41,...,2,2024-02-24,Northridge - Black & Blue Invitational,Outdoor,Finals,2024 Outdoor,Official,field,higher,13.41
25,1,Harley McNeal,Female,12,2024,400m,59.69,59.69,NaN,NaN,...,3,2024-04-26,AHSAA 6A - Section 3 - Northridge,Outdoor,Finals,2024 Outdoor,Official,track,lower,59.69
43,2,Khiyon Lewis,Male,11,2024,triple jump,41-0,NaN,492.00,12.50,...,6,2023-03-11,Vestavia Hills - Varsity - King of the Mountai...,Outdoor,Finals,2023 Outdoor,Official,field,higher,12.50


# Personal Best Detection System



*   Sort the results by athlete, event, and date
*   Log each best performance
*   If the current performance is better than all previous performances then is_pr == True



In [84]:
# Sort dataframe by athlete, event, and date

def detect_pr_progression(group):
    """
    Determines if a result in the dataframe is a personal best based on previous results
    Captures the current PR
    Calculates the improvement of every PR
    """

    group = group.sort_values("date", ascending=True).copy()

    best_so_far = None
    pr_list = [] # Determines if a result in the df is a result based on previous results
    current_pr_list = [] # Captures the current PR
    improvement_list = [] # Calculate the improvement of every pr


    direction = group["better_direction"].iloc[0]

    for result in group["result_value"]:
        if best_so_far == None:
            best_so_far = result
            pr_list.append(True)
            improvement_list.append(None)

        else:
            if direction == "lower": # Track event
                if result < best_so_far: # if result is lower than best so far // PR
                    pr_list.append(True)

                    improvement = best_so_far - result # Calculate the improvement from the last pr
                    improvement_list.append(improvement)

                    best_so_far = result
                else:
                    pr_list.append(False)
                    improvement_list.append(None)

            elif direction == "higher": # Field event
                if result > best_so_far: # if result is higher than best so far // PR
                    pr_list.append(True)

                    improvement = result - best_so_far
                    improvement_list.append(improvement)

                    best_so_far = result
                else:
                    pr_list.append(False)
                    improvement_list.append(None)

        current_pr_list.append(best_so_far)

    group["is_pr"] = pr_list
    group["current_pr"] = current_pr_list
    group["improvement"] = improvement_list

    return group

In [85]:
# updates the df with is_pr column

df = df.groupby(["athlete_id", "event"], as_index=False, observed=True).apply(detect_pr_progression).reset_index(drop=True)

# check the df for bad data

df[
    ["name", "event", "date", "result_value", "is_pr", "current_pr", "improvement"]
].sort_values(["name", "event", "date"])

/tmp/ipykernel_923/1049881494.py:3: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/tmp/ipykernel_923/1049881494.py:3: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,name,event,date,result_value,is_pr,current_pr,improvement
0,Harley McNeal,100m,2023-02-25,12.33,True,12.33,NaN
1,Harley McNeal,100m,2023-03-18,12.36,False,12.33,NaN
2,Harley McNeal,100m,2023-03-25,12.53,False,12.33,NaN
3,Harley McNeal,100m,2023-04-28,11.93,True,11.93,0.4
4,Harley McNeal,100m,2023-04-29,12.26,False,11.93,NaN
...,...,...,...,...,...,...,...
96,Khiyon Lewis,triple jump,2024-04-06,13.43,False,14.76,NaN
97,Khiyon Lewis,triple jump,2024-04-13,13.87,False,14.76,NaN
98,Khiyon Lewis,triple jump,2024-04-19,12.90,False,14.76,NaN
99,Khiyon Lewis,triple jump,2024-04-27,14.15,False,14.76,NaN


In [86]:
dataset = df.groupby(["athlete_id", "event"], as_index=False, observed=True)

groups = [group_df.sort_values("date") for group_key, group_df in dataset]

len(groups)

10

# Athlete Event Progression
1. How has this athlete progressed over time in one event?

In [87]:
#@title Athlete Event Progression

def athlete_event_progression(dataset):
    """    if dataset.name.value_counts() > 1 or dataset.event.value_counts() > 1:
        return "Hey this is wrong"""

    name = dataset["name"].iloc[0]
    event = dataset["event"].iloc[0]
    better_direction = dataset["better_direction"].iloc[0]

    performance_line = px.line(dataset, x="date", y="result_value", markers=True, hover_data=["meet", "raw_result", "placement", "wind", "atmosphere", "race_type"], title=f"Athlete: {name}\n Event: {event.title()}")

    if better_direction == "lower":
        performance_line.update_yaxes(autorange="reversed")
        performance_line.update_layout(xaxis_title="Date", yaxis_title = "Seconds")
    else:
        performance_line.update_layout(xaxis_title="Date", yaxis_title = "Meters")

    performance_line.show()

for group in groups:
    athlete_event_progression(group)



# Current PR Progression Line
1. How did athlete's personal best improve over time

In [88]:
def current_pr_progression_line(dataset):
    dataset = dataset[dataset["is_pr"] == True]

    hover_data = ["meet", "placement", "wind", "atmosphere", "race_type", "raw_result"]

    name = dataset["name"].iloc[0]
    event = dataset["event"].iloc[0]
    better_direction = dataset["better_direction"].iloc[0]

    progression_line = px.line(dataset, x="date", y="result_value", hover_data=hover_data, title=f"{name}'s {event.title()} Progression", markers=True)

    if better_direction == "lower":
        progression_line.update_yaxes(autorange="reversed")
        progression_line.update_layout(xaxis_title="Date", yaxis_title = "Seconds")
    else:
        progression_line.update_layout(xaxis_title="Date", yaxis_title = "Meters")


    progression_line.show()

for group in groups:
    current_pr_progression_line(group)


In [89]:
# @title Medal Count

athlete_list = df.groupby("athlete_id")

def medal_count(athlete_dataset):
    medal_df = athlete_dataset[athlete_dataset["placement"].isin([1, 2, 3])].copy()

    name = athlete_dataset["name"].iloc[0]

    medal_counts = medal_df["placement"].value_counts().sort_index().reset_index()
    medal_counts.columns = ["placement", "count"]

    medal_counts["placement"] = medal_counts["placement"].astype(str)
    medal_counts["medal"] = medal_counts["placement"].map({
        "1" : "Gold",
        "2" : "Silver",
        "3" : "Bronze"
        })

    fig = px.bar(
        medal_counts,
        x="placement",
        y="count",
        color="medal",
        color_discrete_map= {
            "Gold" : "#FFD700",
            "Silver" : "#C0C0C0",
            "Bronze" : "#CE8946"
        },
        title= f"{name}'s High School Career Medal Count"
    )

    fig.show()

for athlete_id, athlete_df in athlete_list:
    medal_count(athlete_df)


# Rank Coached Events by Competitive Performance

*   Here, we will determine what event a coach is best at by evaluating all results for each event in the dataframe and obtaining the lowest average placement for that event.
* This analysis showcases a coach's event range and competitive success across multiple disciplines.

In [90]:
event_df = df.groupby("event", as_index=False, observed=True)

def ranking_coached_events(event_df) -> dict:
    """
    Returns a sorted dictionary of ranked events by average placement across all results for that event
    """
    coached_events = {}

    for event_key, event in event_df:
        performance_count = event.shape[0]
        avg_placement = float(round(event["placement"].mean(), 2))
        event = event["event"].iloc[0]

        coached_events[event] = {"avg_placement" : avg_placement,
                                 "result_count" : performance_count}


    ranked_coaching_events = sorted(coached_events.items(), key=lambda item: item[1]["avg_placement"])

    return ranked_coaching_events

ranked_events = ranking_coached_events(event_df)

ranked_events_df = ranked_events_df = pd.DataFrame([
    {
        "event": event,
        "avg_placement": data["avg_placement"],
        "result_count": data["result_count"]
    }
    for event, data in ranked_events
])

ranked_events_df


,event,avg_placement,result_count
0,400m,2.78,9
1,triple jump,4.27,26
2,200m,5.69,16
3,60m,6.29,7
4,800m,7.00,1
5,100m,8.31,16
6,long jump,9.81,26


In [91]:
## Visualization

ranked_events_df["label"] = ranked_events_df["result_count"].astype(str) + " results"

def chart_coaching_event_rankings(ranked_events_df):

    ranking_bar = px.bar(
        ranked_events_df,
        x="event",
        y="avg_placement",
        color="event",
        hover_data=["result_count"],
        title="Coached Events Ranked by Average Placement",
        labels={
            "event": "Event",
            "avg_placement": "Average Placement",
            "result_count": "Number of Results"
        },
        text="label",
    )

    ranking_bar.update_xaxes(title="Event")
    ranking_bar.update_yaxes(title="Average Placement")


    ranking_bar.show()


chart_coaching_event_rankings(ranked_events_df=ranked_events_df)

# Athlete's Summary

## Showcases the following:

*   Total performances
*   Total medal count
*   Total number of personal best performances
*   Current personal best's across all events
*   The athelete's best event



In [92]:
def athletes_summary(athlete_df):
    """
    Find total performances
    Find medal count -- Add State Medal Count Later
    Find total pr's
    Find current pr's
    Determine best event
    """

    name = athlete_df["name"].iloc[0]
    total_performances = athlete_df.raw_result.count()
    gold_medal_count = athlete_df["placement"].isin([1]).sum()
    silver_medal_count = athlete_df["placement"].isin([2]).sum()
    bronze_medal_count = athlete_df["placement"].isin([3]).sum()
    total_medal_count = gold_medal_count + silver_medal_count + bronze_medal_count
    total_prs = athlete_df["is_pr"].sum()
    current_prs = {}

    event_df = athlete_df.groupby("event", as_index=False, observed=True)

    current_prs = get_prs(event_df) # Grabs the current prs

    main_events = get_main_event(athlete_df)

    summary = {
        "name" : name,
        "total_performances" : total_performances,
        "gold_medal_count" : gold_medal_count,
        "silver_medal_count" : silver_medal_count,
        "bronze_medal_count" : bronze_medal_count,
        "total_medal_count" : total_medal_count,
        "total_prs" : total_prs,
        "current_prs" : current_prs,
        "main_event(s)" : main_events
    }

    return summary

def get_main_event(athlete_df) -> list:
    """
    Returns best event(s) based on average placement
    """
    avg_placement = athlete_df.groupby("event", observed=True)["placement"].mean()

    best_event_value = avg_placement.min()

    best_events = avg_placement[avg_placement == best_event_value].index.tolist()

    return best_events

def get_prs(event_df):
    current_prs = {}

    for event_key, performance in event_df:
        event_type = performance["event_type"].iloc[0]
        event = performance["event"].iloc[0]

        if event_type == "field":
            pr_idx = performance["result_value"].idxmax()
        elif event_type == "track":
            pr_idx = performance["result_value"].idxmin()

        pr = performance.loc[pr_idx, "raw_result"]

        current_prs[event] = pr


    return current_prs


In [93]:
summaries = []
for ath_key, ath_df in athlete_list:
    summaries.append(athletes_summary(ath_df))

In [94]:
summaries

[{'name': 'Harley McNeal',
  'total_performances': np.int64(40),
  'gold_medal_count': np.int64(22),
  'silver_medal_count': np.int64(5),
  'bronze_medal_count': np.int64(3),
  'total_medal_count': np.int64(30),
  'total_prs': np.int64(14),
  'current_prs': {'100m': '11.93',
   '200m': '24.24',
   '400m': '54.97',
   '60m': '7.69',
   '800m': '2:25.13'},
  'main_event(s)': ['400m']},
 {'name': 'Khiyon Lewis',
  'total_performances': np.int64(61),
  'gold_medal_count': np.int64(10),
  'silver_medal_count': np.int64(8),
  'bronze_medal_count': np.int64(5),
  'total_medal_count': np.int64(23),
  'total_prs': np.int64(24),
  'current_prs': {'100m': '11.55',
   '200m': '24.48',
   '400m': '55.2',
   'long jump': '22-11.75',
   'triple jump': '48-5'},
  'main_event(s)': ['triple jump']}]

In [95]:
summaries_df = pd.DataFrame(summaries)

summaries_df.head()

,name,total_performances,gold_medal_count,silver_medal_count,bronze_medal_count,total_medal_count,total_prs,current_prs,main_event(s)
0,Harley McNeal,40,22,5,3,30,14,"{'100m': '11.93', '200m': '24.24', '400m': '54...",[400m]
1,Khiyon Lewis,61,10,8,5,23,24,"{'100m': '11.55', '200m': '24.48', '400m': '55...",[triple jump]
